# Maximum Flow using the Edmonds–Karp Algorithm

## Introduction
A **flow network** is a directed graph in which every edge has a **capacity**, which is the maximum amount of flow it can carry. There is a **source** `s` that produces flow and a **sink** `t` that consumes it. The **maximum flow problem** asks: *what is the largest total amount of flow that can be sent from `s` to `t` without exceeding any edge capacity?*

The **Edmonds–Karp algorithm** is an implementation of the **Ford–Fulkerson method**. Its key idea is that it always augments along the **shortest path** (fewest edges) from `s` to `t` in the **residual graph**, found with **Breadth-First Search (BFS)**. This choice guarantees a polynomial running time.

## Core Concepts
- **Residual graph:** shows how much extra flow can still be pushed along each edge. After sending flow `f` along `u → v`, the residual capacity of `u → v` decreases by `f` and the residual capacity of the reverse edge `v → u` increases by `f`.
- **Augmenting path:** a path from `s` to `t` in the residual graph where every edge has residual capacity > 0.
- **Bottleneck:** the smallest residual capacity on an augmenting path; it is the amount of flow we can push along that path.
- **Termination:** when no augmenting path exists, the flow is maximum (Max-Flow Min-Cut theorem).

## Notebook Structure
1. Import libraries
2. `bfs()`: find a shortest augmenting path
3. `Edmund_Karp()`: repeatedly augment until no path remains
4. Define an example network
5. Run the algorithm and interpret the result
6. Time and space complexity

## 1. Importing Libraries
- **`numpy`** is used to store the capacity matrix as a 2-D array (`np.full` for the parent array and `np.inf` as the initial "infinite" bottleneck).
- **`collections.deque`** is a double-ended queue that gives **O(1)** `append` and `popleft`, which is what BFS needs. A normal Python list would take O(n) for `pop(0)`.

In [21]:
import numpy as np
from collections import deque

## 2. Breadth-First Search: Finding the Shortest Augmenting Path

`bfs(graph, path, s, t)` searches the **residual graph** for the path from `s` to `t` with the fewest edges.

**Parameters**
| Name | Meaning |
|---|---|
| `graph` | Residual capacity matrix, where `graph[u][v]` is the remaining capacity of edge `u → v` |
| `path` | An empty list that is filled with the vertices of the path found (an output parameter) |
| `s`, `t` | Source and sink vertex indices |

**How it works**
1. Put the source in the queue. Create a parent array `p` initialised to `-1`, which means "not visited yet". Set `p[s] = s` so the source counts as visited.
2. Pop a vertex `curr` and look at every vertex `i`. An edge is usable only if `graph[curr][i] > 0` (residual capacity remains).
3. If `i == t`, the sink is reached. Walk back through the `p` array from `curr` to `s`, collect the vertices, and reverse them to get the path `s → ... → t`. Return `True`.
4. Otherwise, if `i` is unvisited (`p[i] == -1`), record `p[i] = curr` and enqueue it.
5. If the queue empties without reaching `t`, no augmenting path exists, so return `False`.

**Why BFS?** BFS explores vertices level by level, so the first time it reaches `t` it has found a path with the **minimum number of edges**. This is exactly what distinguishes Edmonds–Karp from a generic Ford–Fulkerson (which might use DFS and take very long on unlucky inputs).

**Note:** the function returns as soon as `t` is discovered rather than waiting for `t` to be dequeued, which saves some work.

In [22]:
def bfs(graph,path,s,t):
    q = deque()
    q.append(s)
    n = len(graph)
    p = np.full(n,-1)  #parent
    p[s] = s
    while q:
        curr = q.popleft()
        for i in range(n):
           if(graph[curr][i] > 0):
                if(i==t):
                    x = curr
                    path.append(t)
                    while(x!=s):
                         path.append(x)
                         x = p[x]
                    path.append(s)
                    path.reverse()
                    return True
                elif(p[i]==-1):
                    q.append(i)
                    p[i] = curr
    return False


## 3. The Edmonds–Karp Main Routine

`Edmund_Karp(graph, s, t)` repeatedly calls `bfs` and pushes flow along every path it finds.

**Algorithm**
1. Start with `max_flow = 0`.
2. While `bfs` finds an augmenting path:
   1. **Find the bottleneck:** `min_wt` is the minimum residual capacity along the path.
   2. **Update the residual graph** for each edge `(u, v)` on the path:
      - `graph[u][v] -= min_wt`: less capacity remains forward.
      - `graph[v][u] += min_wt`: a reverse edge is created or increased, which lets later iterations "undo" flow if a better routing exists.
   3. Add `min_wt` to `max_flow` and reset `path`.
3. When `bfs` returns `False`, return `max_flow`.

**Important notes**
- The function **modifies `graph` in place** (it turns it into the residual graph). That is why the call below passes `graph.copy()`, so the original capacities are preserved.
- The reverse-edge update is the heart of the algorithm. Without it, the method could get stuck at a non-optimal flow.
- The variable `n = len(graph)` is defined but never used, and it can be removed.
- The conventional spelling is *Edmonds–Karp*. The function name `Edmund_Karp` is just an identifier, so it works either way.

In [ ]:
def Edmonds_Karp (graph,s,t):
    #s is source
    #t is target or sink
    n= len(graph)
    path =[]
    max_flow =0 
    while(bfs(graph,path,s,t)):
        min_wt= np.inf
        for i in range (len(path)-1):
            min_wt = min(graph[path[i]][path[i+1]],min_wt)
        
        for i in range (len(path)-1):
            graph[path[i]][path[i+1]] -= min_wt
            graph[path[i+1]][path[i]] += min_wt
        
        max_flow += min_wt
        path = []
    
    
    return max_flow    
        
    
        

## 4. Example Flow Network

The network is given as an **adjacency (capacity) matrix**, where `graph[i][j]` is the capacity of the directed edge `i → j` (0 means no edge).

| Index | Vertex |
|---|---|
| 0 | S (source) |
| 1 | A |
| 2 | B |
| 3 | C |
| 4 | D |
| 5 | T (sink) |

**Edges (capacities)**
- S → A (16), S → B (13)
- A → B (10), A → C (12)
- B → A (4), B → D (14)
- C → B (9), C → T (20)
- D → C (7), D → T (4)

This is the classic textbook example (as in CLRS), whose known maximum flow is **23**, so it is a good test for our implementation.

In [24]:
graph = np.array([
    [0, 16, 13,  0,  0,  0],  # S
    [0,  0, 10, 12,  0,  0],  # A
    [0,  4,  0,  0, 14,  0],  # B
    [0,  0,  9,  0,  0, 20],  # C
    [0,  0,  0,  7,  0,  4],  # D
    [0,  0,  0,  0,  0,  0]   # T
], dtype=int)

## 5. Running the Algorithm

We call `Edmund_Karp` with source `0` (S) and sink `5` (T). A **copy** of the matrix is passed so the original `graph` remains unchanged and can be reused (for example, to verify the result or run again).

In [ ]:
max_flow = Edmonds_Karp(graph.copy(), 0, 5)
print(f"Max Flow: {max_flow}")

Max Flow: 23


## 6. Result and Walkthrough

**Output:** `Max Flow: 23`

This matches the known answer for this network. Here is how the algorithm reaches it (vertices explored in index order):

| Iteration | Shortest augmenting path | Bottleneck | Total flow |
|---|---|---|---|
| 1 | S → A → C → T | min(16, 12, 20) = **12** | 12 |
| 2 | S → B → D → T | min(13, 14, 4) = **4** | 16 |
| 3 | S → B → D → C → T | min(9, 10, 7, 8) = **7** | 23 |
| 4 | none found | – | **23** (stop) |

BFS is called 4 times in total: 3 successful augmentations and 1 final failed search that proves optimality.

**Verification with the Max-Flow Min-Cut theorem**
After the last iteration, the vertices still reachable from S in the residual graph are {S, A, B, D}. The edges leaving this set towards {C, T} are:
- A → C = 12
- D → C = 7
- D → T = 4

Their total capacity is 12 + 7 + 4 = **23**, which equals the max flow. So this is a **minimum cut**, confirming the answer is optimal.

## 7. Time and Space Complexity

Let **V** be the number of vertices and **E** the number of edges.

### Time Complexity
- **Number of augmentations:** Edmonds–Karp performs at most **O(V · E)** augmenting paths. Each augmentation saturates at least one critical edge, and an edge can become critical only O(V) times because shortest-path distances never decrease.
- **Cost per BFS:**
  - With an adjacency list it is O(E), giving the standard bound **O(V · E²)**.
  - **This notebook uses an adjacency matrix**, so each BFS scans all V entries of every dequeued vertex, costing **O(V²)**.
- **Path update:** O(V) per augmentation, which is negligible compared with BFS.

**Overall for this implementation: O(V · E) × O(V²) = O(V³ · E)**, which is O(V⁵) in the worst case of a dense graph (E ≈ V²). The standard bound for adjacency-list implementations is **O(V · E²)**.

Importantly, this bound is independent of the capacity values, unlike plain Ford–Fulkerson, which can take O(E · max_flow) time.

### Space Complexity
- Capacity / residual matrix: **O(V²)**
- Parent array, BFS queue, and path list: **O(V)** each
- The `graph.copy()` in the calling code adds another O(V²).

**Overall: O(V²)** (O(V + E) if an adjacency list were used instead).

### Summary
| | Adjacency matrix (this notebook) | Adjacency list (standard) |
|---|---|---|
| Time | O(V³ · E) | O(V · E²) |
| Space | O(V²) | O(V + E) |